# Feature Engineering

This notebook prepares the cleaned traffic dataset for machine learning by creating useful time-based and weather-related features.


In [ ]:
import pandas as pd

df = pd.read_csv(r"/content/traffic_cleaned.csv")

df["date_time"] = pd.to_datetime(df["date_time"])

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
df["year"] = df["date_time"].dt.year
df["month"] = df["date_time"].dt.month
df["day"] = df["date_time"].dt.day
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df.head()

In [ ]:
print(df.shape)
print(df.columns.tolist())

In [ ]:
print("Weather Main categories:")
print(df["weather_main"].value_counts())

print("\nWeather Description categories:")
print(df["weather_description"].value_counts())

In [ ]:
df = df.drop(columns=["weather_description"], errors="ignore")

print("Shape after removing weather_description:", df.shape)
print(df.columns.tolist())

In [ ]:
print("Holiday values:")
print(df["holiday"].value_counts(dropna=False))

print("\nUnique holiday values:")
print(df["holiday"].unique())

In [ ]:
if "holiday" in df.columns:
    # Create binary holiday feature
    df["is_holiday"] = df["holiday"].notna().astype(int)
    # Remove original holiday column
    df = df.drop(columns=["holiday"])
    print(df["is_holiday"].value_counts())
elif "is_holiday" in df.columns:
    print("Column 'holiday' was already processed. 'is_holiday' column exists:")
    print(df["is_holiday"].value_counts())
else:
    print("Neither 'holiday' nor 'is_holiday' columns found. Cannot create 'is_holiday'.")

In [ ]:
df = pd.get_dummies(
    df,
    columns=["weather_main"],
    prefix="weather",
    dtype=int
)

print("Shape:", df.shape)
print(df.columns.tolist())

In [ ]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

In [ ]:
duplicates = df[df.duplicated(keep=False)].sort_values("date_time")

print("Number of rows involved in exact duplicates:", len(duplicates))

duplicates.head(20)

In [ ]:
duplicate_counts = (
    df.value_counts()
      .reset_index(name="count")
)

duplicate_counts[duplicate_counts["count"] > 1].head(20)

In [ ]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicate rows removed:", before - after)
print("Remaining duplicate rows:", df.duplicated().sum())

In [ ]:
timestamp_counts = df["date_time"].value_counts()

print("Unique timestamps:", timestamp_counts.size)
print("Timestamps appearing more than once:",
      (timestamp_counts > 1).sum())

print("\nMaximum rows for a single timestamp:",
      timestamp_counts.max())

In [ ]:
duplicate_timestamps = (
    df[df["date_time"].duplicated(keep=False)]
    .sort_values("date_time")
)

duplicate_timestamps[
    ["date_time", "traffic_volume", "temp", "rain_1h",
     "snow_1h", "clouds_all"]
].head(20)

In [ ]:
# Investigate what differs between rows sharing the same timestamp

timestamp_analysis = df.groupby("date_time").agg(
    rows=("date_time", "size"),
    unique_traffic=("traffic_volume", "nunique"),
    unique_temp=("temp", "nunique"),
    unique_rain=("rain_1h", "nunique"),
    unique_snow=("snow_1h", "nunique"),
    unique_clouds=("clouds_all", "nunique")
)

# find timestamps where multiple observations have different values
timestamp_analysis[
    (timestamp_analysis["rows"] > 1) &
    (
        (timestamp_analysis["unique_traffic"] > 1) |
        (timestamp_analysis["unique_temp"] > 1) |
        (timestamp_analysis["unique_rain"] > 1) |
        (timestamp_analysis["unique_snow"] > 1) |
        (timestamp_analysis["unique_clouds"] > 1)
    )
].head(20)

In [ ]:
print("Duplicate timestamps with different traffic values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_traffic"] > 1)).sum())

print("Duplicate timestamps with different temperature values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_temp"] > 1)).sum())

print("Duplicate timestamps with different rain values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_rain"] > 1)).sum())

print("Duplicate timestamps with different snow values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_snow"] > 1)).sum())

print("Duplicate timestamps with different cloud values:",
      ((timestamp_analysis["rows"] > 1) &
       (timestamp_analysis["unique_clouds"] > 1)).sum())

In [ ]:
weather_columns = [
    "weather_Clear",
    "weather_Clouds",
    "weather_Drizzle",
    "weather_Fog",
    "weather_Haze",
    "weather_Mist",
    "weather_Rain",
    "weather_Smoke",
    "weather_Snow",
    "weather_Squall",
    "weather_Thunderstorm"
]

weather_per_timestamp = df.groupby("date_time")[weather_columns].sum()

conflicting_weather = weather_per_timestamp[
    weather_per_timestamp.sum(axis=1) > 1
]

print("Timestamps with multiple weather categories:",
      len(conflicting_weather))

conflicting_weather.head(20)

In [ ]:
weather_features = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all"
]

weather_encoded = [
    "weather_Clear",
    "weather_Clouds",
    "weather_Drizzle",
    "weather_Fog",
    "weather_Haze",
    "weather_Mist",
    "weather_Rain",
    "weather_Smoke",
    "weather_Snow",
    "weather_Squall",
    "weather_Thunderstorm"
]

time_features = [
    "year",
    "month",
    "day",
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday"
]

aggregation_rules = {
    "traffic_volume": "first",
    "temp": "mean",
    "rain_1h": "mean",
    "snow_1h": "mean",
    "clouds_all": "mean",
    "year": "first",
    "month": "first",
    "day": "first",
    "hour": "first",
    "day_of_week": "first",
    "is_weekend": "first",
    "is_holiday": "first"
}

# Average one-hot weather features
for col in weather_encoded:
    aggregation_rules[col] = "mean"

df_aggregated = (
    df.groupby("date_time", as_index=False)
      .agg(aggregation_rules)
)

print("Original rows:", len(df))
print("Aggregated rows:", len(df_aggregated))
print("Unique timestamps:", df["date_time"].nunique())

In [ ]:
print("Duplicate timestamps:", df_aggregated["date_time"].duplicated().sum())
print("Missing values:", df_aggregated.isnull().sum().sum())
print("Shape:", df_aggregated.shape)

In [ ]:
df_aggregated.head()

In [ ]:
df = df_aggregated.copy()

df = df.sort_values("date_time").reset_index(drop=True)

print("Shape:", df.shape)
print("Start:", df["date_time"].min())
print("End:", df["date_time"].max())

In [ ]:
features = [col for col in df.columns
            if col not in ["date_time", "traffic_volume"]]

X = df[features]
y = df["traffic_volume"]

print("Number of features:", len(features))
print("\nFeatures:")
print(features)

print("\nTarget:", y.name)

In [ ]:
df.to_csv("traffic_features.csv", index=False)

print("Feature-engineered dataset saved.")